# The Fashion-MNIST Dataset
In this notebook, we make a few attempts at predicting the classes of different fashion items. Here, we use the [Fashion-MNIST dataset](https://github.com/zalandoresearch/fashion-mnist) which consists of images of items belonging to 10 different classes:

| Label | Description |
| --- | --- |
| 0 | T-shirt/top |
| 1 | Trouser |
| 2 | Pullover |
| 3 | Dress |
| 4 | Coat |
| 5 | Sandal |
| 6 | Shirt |
| 7 | Sneaker |
| 8 | Bag |
| 9 | Ankle boot |

Following is what we'll do to predict the classes:
- First, we'll start with a basic feed-forward neural network
- Second, we'll add a few convolution layers along with pooling
- Third, we'll add regularization in the form of batch normalization & dropout
- Fourth, we'll see if data augmentation/weight decay/scheduled learning rate will lead to improvements

## Imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
# use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


-----
## Helper Functions

In [3]:
def trainer_func(model, criterion, optimizer, train_loader, test_loader, device, epochs=10, scheduler=None):
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)  # needed to use GPU, comment this line if you want to ensure using CPU
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
    
        # Evaluate on test set
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for xb, yb in test_loader:
                outputs = model(xb.to(device))
                _, predicted = torch.max(outputs, 1)
                total += yb.size(0)
                correct += (predicted == yb.to(device)).sum().item()
        acc = correct / total
        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}, Test Acc: {acc:.2f}")

        if scheduler:
            scheduler.step()

In [4]:
def tester_func(model, test_loader, device):
    # predict on test set
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())
    
    # evaluate
    classes = [
        "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
        "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
    ]
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=classes))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    
    return

-----
## Load data

In [5]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # normalize to [-1, 1]
])

train_dataset = torchvision.datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.FashionMNIST(
    root="./data", train=False, download=True, transform=transform
)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64,   shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_dataset,  batch_size=1000, shuffle=False)

-----
## 1. Feed-forward Neural Network

In [6]:
# neural network design
class FashionNet(nn.Module):
    def __init__(self):
        super(FashionNet, self).__init__()
        self.fc1 = nn.Linear(28*28, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(-1, 28*28)  # flatten images
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)  # raw logits
        return x

In [7]:
# train model
model = FashionNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
trainer_func(model, criterion, optimizer, train_loader, test_loader, device, epochs=20)

Epoch 1/20, Loss: 0.4959, Test Acc: 0.85
Epoch 2/20, Loss: 0.3701, Test Acc: 0.86
Epoch 3/20, Loss: 0.3310, Test Acc: 0.87
Epoch 4/20, Loss: 0.3039, Test Acc: 0.87
Epoch 5/20, Loss: 0.2864, Test Acc: 0.87
Epoch 6/20, Loss: 0.2692, Test Acc: 0.87
Epoch 7/20, Loss: 0.2579, Test Acc: 0.88
Epoch 8/20, Loss: 0.2443, Test Acc: 0.88
Epoch 9/20, Loss: 0.2324, Test Acc: 0.89
Epoch 10/20, Loss: 0.2218, Test Acc: 0.89
Epoch 11/20, Loss: 0.2094, Test Acc: 0.89
Epoch 12/20, Loss: 0.2009, Test Acc: 0.88
Epoch 13/20, Loss: 0.1928, Test Acc: 0.88
Epoch 14/20, Loss: 0.1850, Test Acc: 0.89
Epoch 15/20, Loss: 0.1799, Test Acc: 0.89
Epoch 16/20, Loss: 0.1679, Test Acc: 0.89
Epoch 17/20, Loss: 0.1633, Test Acc: 0.89
Epoch 18/20, Loss: 0.1539, Test Acc: 0.89
Epoch 19/20, Loss: 0.1507, Test Acc: 0.88
Epoch 20/20, Loss: 0.1416, Test Acc: 0.89


In [8]:
# predict on test set and evaluate
tester_func(model, test_loader, device)

Accuracy: 0.8904

Classification Report:
              precision    recall  f1-score   support

 T-shirt/top       0.81      0.88      0.84      1000
     Trouser       0.99      0.98      0.98      1000
    Pullover       0.79      0.85      0.82      1000
       Dress       0.92      0.87      0.89      1000
        Coat       0.82      0.83      0.82      1000
      Sandal       0.94      0.97      0.96      1000
       Shirt       0.76      0.68      0.72      1000
     Sneaker       0.94      0.94      0.94      1000
         Bag       0.98      0.96      0.97      1000
  Ankle boot       0.96      0.95      0.96      1000

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000


Confusion Matrix:
[[881   0  14  17   5   5  72   0   6   0]
 [  4 979   0  12   3   0   2   0   0   0]
 [ 26   2 848   4  54   1  63   0   2   0]
 [ 35   5  24 867  45   2  21   0   1   0]
 [  1   2

We got an overall accuracy of > 85%. 

It seems the model confuses these together:
- shirts & t-shirts
- coats & pullovers
- dresses & shirts/t-shirts

Let's see if we can improve on this.

-----
## 2. Convolution + Pooling

In [9]:
# neural network design
class FashionCNN(nn.Module):
    def __init__(self):
        super(FashionCNN, self).__init__()
        # Conv layer 1: input=1 channel, output=32 filters, kernel=3x3
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        # Conv layer 2: input=32, output=64 filters
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        # Pooling layer (2x2 max pooling)
        self.pool = nn.MaxPool2d(2, 2)
        # Fully connected layers
        self.fc1 = nn.Linear(64 * 7 * 7, 128)  # 28→14→7 after pooling
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        # Conv + ReLU + Pool
        x = F.relu(self.pool(self.conv1(x)))  # [1x28x28] -> [32x14x14]
        x = F.relu(self.pool(self.conv2(x)))  # [32x14x14] -> [64x7x7]
        # Flatten
        x = x.view(-1, 64 * 7 * 7)
        # FC layers
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [10]:
# train model
model = FashionCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
trainer_func(model, criterion, optimizer, train_loader, test_loader, device, epochs=20)

Epoch 1/20, Loss: 0.4336, Test Acc: 0.88
Epoch 2/20, Loss: 0.2757, Test Acc: 0.88
Epoch 3/20, Loss: 0.2328, Test Acc: 0.90
Epoch 4/20, Loss: 0.1976, Test Acc: 0.91
Epoch 5/20, Loss: 0.1721, Test Acc: 0.92
Epoch 6/20, Loss: 0.1501, Test Acc: 0.92
Epoch 7/20, Loss: 0.1272, Test Acc: 0.92
Epoch 8/20, Loss: 0.1107, Test Acc: 0.92
Epoch 9/20, Loss: 0.0934, Test Acc: 0.91
Epoch 10/20, Loss: 0.0790, Test Acc: 0.92
Epoch 11/20, Loss: 0.0669, Test Acc: 0.92
Epoch 12/20, Loss: 0.0556, Test Acc: 0.92
Epoch 13/20, Loss: 0.0497, Test Acc: 0.92
Epoch 14/20, Loss: 0.0415, Test Acc: 0.92
Epoch 15/20, Loss: 0.0348, Test Acc: 0.91
Epoch 16/20, Loss: 0.0308, Test Acc: 0.92
Epoch 17/20, Loss: 0.0276, Test Acc: 0.91
Epoch 18/20, Loss: 0.0300, Test Acc: 0.92
Epoch 19/20, Loss: 0.0227, Test Acc: 0.91
Epoch 20/20, Loss: 0.0223, Test Acc: 0.92


In [11]:
# predict on test set and evaluate
tester_func(model, test_loader, device)

Accuracy: 0.9187

Classification Report:
              precision    recall  f1-score   support

 T-shirt/top       0.83      0.89      0.86      1000
     Trouser       0.98      0.99      0.98      1000
    Pullover       0.88      0.88      0.88      1000
       Dress       0.90      0.92      0.91      1000
        Coat       0.88      0.87      0.88      1000
      Sandal       0.99      0.98      0.98      1000
       Shirt       0.83      0.73      0.78      1000
     Sneaker       0.96      0.97      0.96      1000
         Bag       0.98      0.98      0.98      1000
  Ankle boot       0.97      0.97      0.97      1000

    accuracy                           0.92     10000
   macro avg       0.92      0.92      0.92     10000
weighted avg       0.92      0.92      0.92     10000


Confusion Matrix:
[[891   0  14  15   2   0  73   0   5   0]
 [  2 990   0   5   1   0   1   0   1   0]
 [ 30   1 882  10  39   0  36   0   2   0]
 [ 29  14   5 922  14   0  15   0   1   0]
 [  4   1

We got an overall accuracy of > 90%. 

While accuracy did improve, it seems the model mispredicts a lot of items as shirts AND mispredicts a lot of shirts as something else!

Let's see if we can improve on this further still.

-----
## 3. Dropout + BatchNorm

In [12]:
# neural network design
class FashionCNNImproved(nn.Module):
    def __init__(self):
        super(FashionCNNImproved, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.bn1(self.pool(self.conv1(x))))  # -> [32, 14, 14]
        x = F.relu(self.bn2(self.pool(self.conv2(x))))  # -> [64, 7, 7]
        x = x.view(-1, 64 * 7 * 7)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)  # dropout on FC layer
        x = self.fc2(x)
        return x

In [13]:
# train model
model = FashionCNNImproved().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
trainer_func(model, criterion, optimizer, train_loader, test_loader, device, epochs=20)

Epoch 1/20, Loss: 0.3798, Test Acc: 0.90
Epoch 2/20, Loss: 0.2595, Test Acc: 0.91
Epoch 3/20, Loss: 0.2163, Test Acc: 0.91
Epoch 4/20, Loss: 0.1911, Test Acc: 0.91
Epoch 5/20, Loss: 0.1676, Test Acc: 0.92
Epoch 6/20, Loss: 0.1477, Test Acc: 0.92
Epoch 7/20, Loss: 0.1319, Test Acc: 0.92
Epoch 8/20, Loss: 0.1199, Test Acc: 0.92
Epoch 9/20, Loss: 0.1071, Test Acc: 0.92
Epoch 10/20, Loss: 0.0948, Test Acc: 0.92
Epoch 11/20, Loss: 0.0867, Test Acc: 0.92
Epoch 12/20, Loss: 0.0795, Test Acc: 0.92
Epoch 13/20, Loss: 0.0705, Test Acc: 0.92
Epoch 14/20, Loss: 0.0650, Test Acc: 0.92
Epoch 15/20, Loss: 0.0628, Test Acc: 0.92
Epoch 16/20, Loss: 0.0564, Test Acc: 0.92
Epoch 17/20, Loss: 0.0550, Test Acc: 0.92
Epoch 18/20, Loss: 0.0524, Test Acc: 0.92
Epoch 19/20, Loss: 0.0476, Test Acc: 0.93
Epoch 20/20, Loss: 0.0436, Test Acc: 0.92


In [14]:
# predict on test set and evaluate
tester_func(model, test_loader, device)

Accuracy: 0.9246

Classification Report:
              precision    recall  f1-score   support

 T-shirt/top       0.87      0.88      0.88      1000
     Trouser       0.99      0.99      0.99      1000
    Pullover       0.88      0.86      0.87      1000
       Dress       0.92      0.95      0.93      1000
        Coat       0.88      0.87      0.88      1000
      Sandal       0.99      0.98      0.99      1000
       Shirt       0.78      0.77      0.78      1000
     Sneaker       0.96      0.97      0.97      1000
         Bag       0.98      0.99      0.99      1000
  Ankle boot       0.97      0.97      0.97      1000

    accuracy                           0.92     10000
   macro avg       0.92      0.92      0.92     10000
weighted avg       0.92      0.92      0.92     10000


Confusion Matrix:
[[884   1  18  14   4   1  75   0   3   0]
 [  0 987   0  10   0   0   1   0   2   0]
 [ 17   1 864   8  43   0  66   0   1   0]
 [ 12   2   7 946  11   0  21   0   1   0]
 [  0   0

Well this model did do slightly better than the previous one (in terms of accuracy).

We'll give this one more shot!

-----
## 4. Data Augmentation + Learning Rate Scheduling + Weight Decay

In [15]:
# data augmentation

transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),      # random flip
    transforms.RandomRotation(10),          # random rotation ±10°
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))    # normalize to [-1,1]
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = torchvision.datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=transform_train
)
# test_dataset = torchvision.datasets.FashionMNIST(
#     root="./data", train=False, download=True, transform=transform_test
# )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
# test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1000, shuffle=False)

In [16]:
# train model
model = FashionCNNImproved().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)  # weight decay
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)  # learning rate scheduling
trainer_func(model, criterion, optimizer, train_loader, test_loader, device, epochs=20, scheduler=scheduler)

Epoch 1/20, Loss: 0.4528, Test Acc: 0.88
Epoch 2/20, Loss: 0.3325, Test Acc: 0.90
Epoch 3/20, Loss: 0.2993, Test Acc: 0.90
Epoch 4/20, Loss: 0.2786, Test Acc: 0.90
Epoch 5/20, Loss: 0.2624, Test Acc: 0.91
Epoch 6/20, Loss: 0.2540, Test Acc: 0.92
Epoch 7/20, Loss: 0.2435, Test Acc: 0.92
Epoch 8/20, Loss: 0.2361, Test Acc: 0.91
Epoch 9/20, Loss: 0.2302, Test Acc: 0.92
Epoch 10/20, Loss: 0.2257, Test Acc: 0.92
Epoch 11/20, Loss: 0.2009, Test Acc: 0.92
Epoch 12/20, Loss: 0.1942, Test Acc: 0.92
Epoch 13/20, Loss: 0.1907, Test Acc: 0.92
Epoch 14/20, Loss: 0.1846, Test Acc: 0.93
Epoch 15/20, Loss: 0.1833, Test Acc: 0.93
Epoch 16/20, Loss: 0.1805, Test Acc: 0.93
Epoch 17/20, Loss: 0.1757, Test Acc: 0.92
Epoch 18/20, Loss: 0.1763, Test Acc: 0.93
Epoch 19/20, Loss: 0.1746, Test Acc: 0.93
Epoch 20/20, Loss: 0.1729, Test Acc: 0.92


In [17]:
# predict on test set and evaluate
tester_func(model, test_loader, device)

Accuracy: 0.9179

Classification Report:
              precision    recall  f1-score   support

 T-shirt/top       0.86      0.91      0.88      1000
     Trouser       0.99      0.99      0.99      1000
    Pullover       0.77      0.95      0.85      1000
       Dress       0.95      0.90      0.92      1000
        Coat       0.90      0.80      0.85      1000
      Sandal       0.99      0.98      0.98      1000
       Shirt       0.83      0.72      0.77      1000
     Sneaker       0.95      0.98      0.96      1000
         Bag       0.99      0.99      0.99      1000
  Ankle boot       0.97      0.96      0.97      1000

    accuracy                           0.92     10000
   macro avg       0.92      0.92      0.92     10000
weighted avg       0.92      0.92      0.92     10000


Confusion Matrix:
[[912   1  23   6   0   1  55   0   2   0]
 [  1 989   0   6   0   0   2   0   2   0]
 [ 17   0 951   5  12   0  15   0   0   0]
 [ 21   8  22 902  23   0  24   0   0   0]
 [  0   0

-----
## What I was working with

In [18]:
print(torch.version.cuda)
print(torch.cuda.is_available())
device_id = torch.cuda.current_device()
print(device_id)
print(torch.cuda.get_device_name(device_id))

12.4
True
0
NVIDIA GeForce RTX 3070 Laptop GPU
